<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/AE_10a_build_sequence_failure_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10A - Sequence Model for Failure Prediction

**Purpose:** Replace the per-snapshot Random Forest (09A) with a sequence
model (GRU) that sees each engine's **trailing history of snapshots**, not
just one snapshot's feature vector in isolation. The panel already encodes
some trend information by hand (`RATE_PER_DAY`, `DAYS_SINCE_LAST`), but a
sequence model can learn escalation/de-escalation patterns across snapshots
directly, which is the reasoning behind moving to sequence models.

**Builds on the real 09A run:** this notebook loads
`ecfr_failure_prediction_panel.parquet` from your production run (1,277,405
rows, 1,621 engines, 188 ever-fail). It carries forward the same discipline:
group-aware (by engine) splitting, natural class imbalance kept in
validation/test, no leakage.

**Two fixes worth calling out before this runs on your data:**
- **`DAYS_SINCE_LAST` sentinel fill.** 09A's baseline filled all NaNs
  (including `DAYS_SINCE_LAST`) with the column median. For an engine that
  has *never* had a given event type, that's wrong — the median plugs in a
  plausible-looking "it happened recently" value. Here, count/rate columns
  are zero-filled (correct: no events = 0) and `DAYS_SINCE_LAST` columns are
  filled with a large sentinel (correct: "no such event in this engine's
  observed history").
- **`ignition_record` / `engine_redline` low time-confidence.** From your
  09A run, these sections had 3.7% and 0.7% timestamp confidence
  respectively. Their features will be mostly sentinel/zero here for the
  same reason they were thin in the RF importances — this notebook doesn't
  fix that data-quality gap, it just doesn't hide it.

**Class imbalance:** `FAILS_WITHIN_7D` positive rate was 0.01% in your run.
Training directly on that is impractical (the model has almost nothing to
learn from per batch), so this notebook subsamples **negative training
rows only** to a configurable ratio. Validation and test sets keep the
natural imbalance, so the reported metrics are honest.

Run cell-by-cell and paste each cell's output back into chat, same as
before.


In [ ]:
# CELL 01 - Setup
from pathlib import Path
import warnings, json, math, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

# --- reuse the exact paths 09A wrote to ---
ROOT = Path('/raid3/e296408/All_ECFRs_working')
BRANCH_A_DIR = ROOT / 'branch_a_ecfr_only'
PANEL_DIR = BRANCH_A_DIR / 'failure_prediction_panel'
PANEL_FILE = PANEL_DIR / 'ecfr_failure_prediction_panel.parquet'

SEQUENCE_DIR = PANEL_DIR / 'sequence_model'
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

# --- config ---
HORIZON_DAYS_LIST = [7, 14, 30]
SEQ_LEN = 12                 # trailing snapshots per sample (12 x 7-day steps ~ 3 months)
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15     # remainder -> test
NEG_SUBSAMPLE_PER_POS = 20   # training-only: keep at most this many negative rows per positive row, per horizon union
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.1
BATCH_SIZE = 256
MAX_EPOCHS = 40
PATIENCE = 6                 # early stop if val avg PR-AUC doesn't improve for this many epochs
LR = 1e-3
SEED = 42
DAYS_SINCE_LAST_SENTINEL = 9999.0   # "never observed in this engine's history"

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def banner(title):
    print('=' * 100); print(title); print('=' * 100)

banner('10A SETUP')
print('PANEL_FILE     :', PANEL_FILE, '(exists=%s)' % PANEL_FILE.exists())
print('SEQUENCE_DIR   :', SEQUENCE_DIR)
print('Device         :', DEVICE)
print('SEQ_LEN        :', SEQ_LEN, ' (trailing snapshots per sample)')
print('Horizons       :', HORIZON_DAYS_LIST)


ModuleNotFoundError: No module named 'torch'

In [ ]:
# CELL 02 - Load the prediction panel built by 09A
if not PANEL_FILE.exists():
    raise FileNotFoundError('Run 09A first -- ecfr_failure_prediction_panel.parquet not found.')

panel = pd.read_parquet(PANEL_FILE)
panel = panel.sort_values(['ENGINE_SERIAL', 'SNAPSHOT_TIME']).reset_index(drop=True)

label_cols = [f'FAILS_WITHIN_{h}D' for h in HORIZON_DAYS_LIST]
missing = [c for c in label_cols + ['ENGINE_SERIAL', 'SNAPSHOT_TIME', 'ENGINE_EVER_FAILS_IN_RECORD'] if c not in panel.columns]
if missing:
    raise ValueError(f'Panel is missing expected columns: {missing}')

banner('PANEL LOADED')
print('Rows                 :', len(panel))
print('Engines               :', panel['ENGINE_SERIAL'].nunique())
for h, c in zip(HORIZON_DAYS_LIST, label_cols):
    print(f'{c}: positive={int(panel[c].sum())} ({panel[c].mean():.4%})')
display(panel.head(3))


### Feature preparation

Count/rate columns (`__COUNT_LOOKBACK`, `__RATE_PER_DAY`, `TOTAL_EVENTS_LOOKBACK`) are
zero-filled — no events in the window genuinely means zero. `__DAYS_SINCE_LAST` columns
are filled with a large sentinel, not the column median, so "never happened for this
engine" doesn't get disguised as "happened recently."


In [ ]:
# CELL 03 - Feature columns and correct NaN handling
non_feature_cols = {'ENGINE_SERIAL', 'SNAPSHOT_TIME', 'ENGINE_EVER_FAILS_IN_RECORD'} | set(label_cols)
feature_cols = [c for c in panel.columns if c not in non_feature_cols]

days_since_cols = [c for c in feature_cols if c.endswith('__DAYS_SINCE_LAST')]
other_cols = [c for c in feature_cols if c not in days_since_cols]

panel[other_cols] = panel[other_cols].fillna(0.0)
panel[days_since_cols] = panel[days_since_cols].fillna(DAYS_SINCE_LAST_SENTINEL)

banner('FEATURE COLUMNS')
print('Total feature columns   :', len(feature_cols))
print('  zero-filled (count/rate/total):', len(other_cols))
print('  sentinel-filled (days-since)  :', len(days_since_cols))
print(days_since_cols)


### Build sequences

One sample per panel row: the trailing up-to-`SEQ_LEN` snapshots ending at that row
(inclusive), right-padded with zeros to `SEQ_LEN`, plus the true length. This mirrors
09A's per-snapshot label exactly — only the *input* changes, from one feature vector
to a short history of them.


In [ ]:
# CELL 04 - Build sequences (one per panel row, per engine's ordered history)
def build_sequences_for_engine(g, feature_cols, seq_len):
    feats = g[feature_cols].to_numpy(dtype=np.float32)
    n = len(g)
    n_features = len(feature_cols)
    out_X = np.zeros((n, seq_len, n_features), dtype=np.float32)
    out_len = np.zeros(n, dtype=np.int64)
    for i in range(n):
        start = max(0, i - seq_len + 1)
        window = feats[start:i + 1]
        length = window.shape[0]
        out_X[i, :length] = window
        out_len[i] = length
    return out_X, out_len

t0 = time.time()
X_chunks, len_chunks, engine_chunks, row_index_chunks = [], [], [], []
for engine, g in panel.groupby('ENGINE_SERIAL', sort=False):
    Xg, Lg = build_sequences_for_engine(g, feature_cols, SEQ_LEN)
    X_chunks.append(Xg)
    len_chunks.append(Lg)
    engine_chunks.append(np.full(len(g), engine))
    row_index_chunks.append(g.index.to_numpy())

X_all = np.concatenate(X_chunks, axis=0)
lengths_all = np.concatenate(len_chunks, axis=0)
engines_all = np.concatenate(engine_chunks, axis=0)
row_index_all = np.concatenate(row_index_chunks, axis=0)

banner('SEQUENCES BUILT')
print('X_all shape      :', X_all.shape, ' (n_samples, seq_len, n_features)')
print('Build time        : %.1fs' % (time.time() - t0))
assert X_all.shape[0] == len(panel)
assert (row_index_all == panel.index.to_numpy()).all(), 'row order mismatch between X_all and panel'


### Engine-level, fail-stratified split

Split **engines**, not rows — an engine's snapshots must all land in the same split,
or the model leaks by learning that specific engine's baseline behavior from
training and getting tested on its own later history. Splitting is stratified by
whether the engine ever fails in-record, since only 188 of 1,621 engines do —
a plain random split of engines risks a val/test fold with too few (or zero)
positive engines to evaluate against.


In [ ]:
# CELL 05 - Engine-level, fail-stratified train/val/test split
ever_fails = panel.groupby('ENGINE_SERIAL')['ENGINE_EVER_FAILS_IN_RECORD'].first().to_dict()

def split_engines(ever_fails, train_frac, val_frac, seed):
    rng = np.random.RandomState(seed)
    engines = np.array(list(ever_fails.keys()))
    labels = np.array([ever_fails[e] for e in engines])
    splits = {'train': [], 'val': [], 'test': []}
    for lab in [0, 1]:
        pool = engines[labels == lab].copy()
        rng.shuffle(pool)
        n = len(pool)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)
        splits['train'].extend(pool[:n_train])
        splits['val'].extend(pool[n_train:n_train + n_val])
        splits['test'].extend(pool[n_train + n_val:])
    return {k: set(v) for k, v in splits.items()}

engine_splits = split_engines(ever_fails, TRAIN_FRAC, VAL_FRAC, SEED)
assert not (engine_splits['train'] & engine_splits['val'])
assert not (engine_splits['train'] & engine_splits['test'])
assert not (engine_splits['val'] & engine_splits['test'])

split_of_row = np.array([
    'train' if e in engine_splits['train'] else ('val' if e in engine_splits['val'] else 'test')
    for e in engines_all
])

banner('ENGINE-LEVEL SPLIT')
for split in ['train', 'val', 'test']:
    engs = engine_splits[split]
    n_fail_engs = sum(ever_fails[e] for e in engs)
    n_rows = int((split_of_row == split).sum())
    print(f'{split:>5}: {len(engs):>5} engines ({n_fail_engs} ever-fail) | {n_rows:>9,} rows')


### Feature scaling (train-only fit) and training-set negative subsampling

The scaler is fit **only on real (non-padded) timesteps from the training split** —
fitting on validation/test, or on the zero-padding, would leak information into the
normalization. Padded positions get scaled too (so they're no longer exactly zero),
but that's harmless: `pack_padded_sequence` excludes them from the GRU's computation
entirely regardless of their value.

Negative rows are subsampled **for training only**, at up to
`NEG_SUBSAMPLE_PER_POS` negatives per positive (union across all three horizons, so
a row that's positive for any horizon is always kept). Validation and test are left
at the true 0.01–0.05% positive rate for an honest read on real-world performance.


In [ ]:
# CELL 06 - Scale features (train-only fit) and build training subsample
n_features = len(feature_cols)
train_mask = split_of_row == 'train'
val_mask = split_of_row == 'val'
test_mask = split_of_row == 'test'

# Fit scaler on real (non-padded) train timesteps only
train_real_steps = []
for i in np.where(train_mask)[0]:
    L = lengths_all[i]
    train_real_steps.append(X_all[i, :L])
train_real_steps = np.concatenate(train_real_steps, axis=0)

scaler = StandardScaler()
scaler.fit(train_real_steps)

def scale_sequences(X):
    n, s, f = X.shape
    flat = X.reshape(-1, f)
    flat_scaled = scaler.transform(flat)
    return flat_scaled.reshape(n, s, f).astype(np.float32)

X_scaled = scale_sequences(X_all)

labels_all = panel[label_cols].to_numpy(dtype=np.float32)  # (n_samples, n_horizons)
any_positive = labels_all.max(axis=1) > 0

# Training subsample: keep all positive rows, subsample negatives
train_idx = np.where(train_mask)[0]
train_pos_idx = train_idx[any_positive[train_idx]]
train_neg_idx = train_idx[~any_positive[train_idx]]

rng = np.random.RandomState(SEED)
n_neg_keep = min(len(train_neg_idx), len(train_pos_idx) * NEG_SUBSAMPLE_PER_POS)
train_neg_keep = rng.choice(train_neg_idx, size=n_neg_keep, replace=False) if n_neg_keep > 0 else train_neg_idx
train_sample_idx = np.concatenate([train_pos_idx, train_neg_keep])
rng.shuffle(train_sample_idx)

val_idx = np.where(val_mask)[0]
test_idx = np.where(test_mask)[0]

banner('SCALING + TRAINING SUBSAMPLE')
print('Train rows (all)         :', len(train_idx))
print('  positive (any horizon) :', len(train_pos_idx))
print('  negative kept for train:', len(train_neg_keep), f'(ratio {NEG_SUBSAMPLE_PER_POS}:1 target)')
print('Train rows used this run :', len(train_sample_idx))
print('Val rows (natural)       :', len(val_idx))
print('Test rows (natural)      :', len(test_idx))


In [ ]:
# CELL 07 - PyTorch Dataset / DataLoader
class SequenceDataset(Dataset):
    def __init__(self, X, lengths, labels, indices):
        self.X = X[indices]
        self.lengths = lengths[indices]
        self.labels = labels[indices]

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, i):
        return (torch.from_numpy(self.X[i]), torch.tensor(self.lengths[i]),
                torch.from_numpy(self.labels[i]))

train_ds = SequenceDataset(X_scaled, lengths_all, labels_all, train_sample_idx)
val_ds = SequenceDataset(X_scaled, lengths_all, labels_all, val_idx)
test_ds = SequenceDataset(X_scaled, lengths_all, labels_all, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

banner('DATALOADERS READY')
print('Train batches:', len(train_loader), ' Val batches:', len(val_loader), ' Test batches:', len(test_loader))


### Model: masked GRU encoder, multi-horizon head

A single GRU encodes each engine's trailing snapshot history; the final hidden state
(at each sequence's *true* last real step, not the padded end) feeds a linear layer
producing one logit per horizon. `pack_padded_sequence` is what makes "true last
step" correct regardless of how much padding a shorter-history engine has.


In [ ]:
# CELL 08 - Model definition
class SequenceFailureModel(nn.Module):
    def __init__(self, n_features, hidden_size, n_horizons, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden_size, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Linear(hidden_size, n_horizons)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        last_hidden = h_n[-1]
        return self.head(last_hidden)

model = SequenceFailureModel(n_features=n_features, hidden_size=HIDDEN_SIZE,
                              n_horizons=len(HORIZON_DAYS_LIST), num_layers=NUM_LAYERS,
                              dropout=DROPOUT).to(DEVICE)

# Per-horizon pos_weight computed from the (subsampled) training set actually used
train_labels_used = labels_all[train_sample_idx]
pos_weight = torch.tensor(
    [(train_labels_used[:, j] == 0).sum() / max((train_labels_used[:, j] == 1).sum(), 1)
     for j in range(len(HORIZON_DAYS_LIST))], dtype=torch.float32, device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

banner('MODEL READY')
print(model)
print('pos_weight per horizon:', pos_weight.tolist())


In [ ]:
# CELL 09 - Training loop with early stopping on validation avg PR-AUC
def evaluate(loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for xb, lb, yb in loader:
            xb, lb = xb.to(DEVICE), lb.to(DEVICE)
            logits = model(xb, lb)
            all_logits.append(logits.cpu().numpy())
            all_labels.append(yb.numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = 1 / (1 + np.exp(-logits))
    metrics = {}
    for j, h in enumerate(HORIZON_DAYS_LIST):
        y = labels[:, j]
        p = probs[:, j]
        if y.sum() == 0 or y.sum() == len(y):
            metrics[h] = {'roc_auc': float('nan'), 'pr_auc': float('nan')}
            continue
        metrics[h] = {'roc_auc': roc_auc_score(y, p), 'pr_auc': average_precision_score(y, p)}
    return metrics, probs, labels

best_val_score = -1
best_state = None
epochs_without_improve = 0
history = []

banner('TRAINING')
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for xb, lb, yb in train_loader:
        xb, lb, yb = xb.to(DEVICE), lb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb, lb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    val_metrics, _, _ = evaluate(val_loader)
    val_pr_aucs = [m['pr_auc'] for m in val_metrics.values() if not math.isnan(m['pr_auc'])]
    val_avg_pr_auc = float(np.mean(val_pr_aucs)) if val_pr_aucs else float('nan')

    history.append({'epoch': epoch, 'train_loss': epoch_loss / n_batches, 'val_avg_pr_auc': val_avg_pr_auc})
    print(f"epoch {epoch:>3}  train_loss={epoch_loss / n_batches:.4f}  "
          f"val_avg_PR-AUC={val_avg_pr_auc:.4f}  "
          + '  '.join(f"{h}D(ROC={m['roc_auc']:.3f},PR={m['pr_auc']:.3f})" for h, m in val_metrics.items()))

    if val_avg_pr_auc > best_val_score:
        best_val_score = val_avg_pr_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improve = 0
    else:
        epochs_without_improve += 1
        if epochs_without_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs).')
            break

model.load_state_dict(best_state)
banner('TRAINING COMPLETE')
print(f'Best validation avg PR-AUC: {best_val_score:.4f}')


### Held-out test evaluation

Test set is at the **natural** imbalance (never subsampled), so these numbers are
the honest estimate of real-world performance. For reference, your 09A Random
Forest baseline scored ROC-AUC 0.881 / 0.880 / 0.869 and PR-AUC 0.106 / 0.128 /
0.064 for the 7/14/30-day horizons -- compare against that below, not against 1.0.


In [ ]:
# CELL 10 - Held-out test evaluation + threshold sensitivity
test_metrics, test_probs, test_labels = evaluate(test_loader)

banner('TEST SET PERFORMANCE (sequence model, natural imbalance)')
summary_rows = []
for h, m in test_metrics.items():
    print(f"{h}D  -- ROC-AUC={m['roc_auc']:.3f}  PR-AUC={m['pr_auc']:.3f}")
    summary_rows.append({'HORIZON_DAYS': h, 'ROC_AUC': m['roc_auc'], 'PR_AUC': m['pr_auc']})

print()
print('For reference -- 09A Random Forest baseline (same test discipline, GroupKFold by engine):')
print('  7D  -- ROC-AUC=0.881  PR-AUC=0.106')
print('  14D -- ROC-AUC=0.880  PR-AUC=0.128')
print('  30D -- ROC-AUC=0.869  PR-AUC=0.064')

# Confusion matrix at 0.5 AND at a recall-oriented threshold, since 0.5 is a poor
# default under this much class imbalance (09A saw the same issue).
print()
banner('CONFUSION MATRICES: default 0.5 vs. recall-tuned threshold')
threshold_rows = []
for j, h in enumerate(HORIZON_DAYS_LIST):
    y = test_labels[:, j]
    p = test_probs[:, j]
    if y.sum() == 0:
        continue
    cm_05 = confusion_matrix(y, (p >= 0.5).astype(int))

    # pick the lowest threshold that still catches ~80% of positives
    order = np.argsort(-p)
    sorted_y = y[order]
    sorted_p = p[order]
    cum_recall = np.cumsum(sorted_y) / sorted_y.sum()
    idx80 = np.searchsorted(cum_recall, 0.80)
    idx80 = min(idx80, len(sorted_p) - 1)
    thresh80 = sorted_p[idx80]
    cm_80r = confusion_matrix(y, (p >= thresh80).astype(int))

    print(f'--- {h}D ---')
    print(f'  @0.50           : {cm_05.tolist()}')
    print(f'  @{thresh80:.4f} (~80% recall): {cm_80r.tolist()}')
    threshold_rows.append({'HORIZON_DAYS': h, 'THRESHOLD_80_RECALL': float(thresh80),
                            'CM_AT_0.5': cm_05.tolist(), 'CM_AT_80_RECALL': cm_80r.tolist()})

pd.DataFrame(summary_rows).to_csv(SEQUENCE_DIR / 'test_performance_summary.csv', index=False)
pd.DataFrame(threshold_rows).to_csv(SEQUENCE_DIR / 'threshold_sensitivity.csv', index=False)


In [ ]:
# CELL 11 - Save model, scaler, config, and test predictions
import pickle

torch.save(best_state, SEQUENCE_DIR / 'sequence_model_state_dict.pt')

with open(SEQUENCE_DIR / 'feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

config = {
    'feature_cols': feature_cols,
    'days_since_cols': days_since_cols,
    'SEQ_LEN': SEQ_LEN,
    'HORIZON_DAYS_LIST': HORIZON_DAYS_LIST,
    'HIDDEN_SIZE': HIDDEN_SIZE,
    'NUM_LAYERS': NUM_LAYERS,
    'DROPOUT': DROPOUT,
    'DAYS_SINCE_LAST_SENTINEL': DAYS_SINCE_LAST_SENTINEL,
}
with open(SEQUENCE_DIR / 'model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

test_pred_df = pd.DataFrame({
    'ENGINE_SERIAL': engines_all[test_idx],
    'SNAPSHOT_TIME': panel.loc[row_index_all[test_idx], 'SNAPSHOT_TIME'].values,
})
for j, h in enumerate(HORIZON_DAYS_LIST):
    test_pred_df[f'LABEL_{h}D'] = test_labels[:, j]
    test_pred_df[f'PRED_PROB_{h}D'] = test_probs[:, j]
test_pred_df.to_csv(SEQUENCE_DIR / 'test_predictions.csv', index=False)

banner('SAVED')
print('Model state dict :', SEQUENCE_DIR / 'sequence_model_state_dict.pt')
print('Scaler           :', SEQUENCE_DIR / 'feature_scaler.pkl')
print('Config           :', SEQUENCE_DIR / 'model_config.json')
print('Test predictions :', SEQUENCE_DIR / 'test_predictions.csv')
print('Performance/threshold CSVs also saved to', SEQUENCE_DIR)
